# Phase 0 — FastV baseline reproduction on Kaggle 2x T4

**Scope:** set up LLaVA-1.5-7B (HF checkpoint), install FastV's patched transformers, and reproduce
FastV's accuracy-vs-pruning-ratio numbers on a subset of **POPE** first (self-contained, no gated
downloads), then optionally **GQA** (needs a larger image download).

**Before you run:** in Kaggle, set Accelerator = GPU T4 x2, and Settings → Internet = On.

**How to use this notebook:** run cells top to bottom. The smoke-test cell (Section 2) is the
important checkpoint — if it errors or the FastV output looks like garbage, **stop and paste the
full traceback back to Claude** before spending GPU-hours on the full eval. I built and grounded
every cell here against the actual FastV/POPE repos, but I can't execute this myself (no GPU in my
environment), so this is a best-effort first pass, not a guaranteed-working script.

**Known risk spots** (flagged so you know what to look at if something breaks):
- `attn_implementation="eager"` is required for FastV's attention-based pruning to work, but it's
  slower and more memory-hungry than SDPA/FlashAttention — this is why we load in 8-bit.
- We install FastV's *own* patched transformers fork (pinned to an early-2024-era version) rather
  than fighting with whatever transformers version Kaggle ships by default, to avoid API drift in
  `LlamaModel` internals that FastV's patch relies on.


## 1. Environment check

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}, {props.total_memory/1e9:.1f} GB, compute capability {props.major}.{props.minor}")


## 2. Install FastV + patched transformers

FastV ships a small patch on top of `transformers==4.39.0.dev0` (adds `fastv_forward` to
`LlamaModel`/`LlamaForCausalLM` and wires it through `LlavaForConditionalGeneration`). We install
their fork in editable mode with `--no-deps` so it doesn't try to drag in a different torch build
than the one Kaggle already has matched to its CUDA driver.

If this cell fails on the `import transformers` line, it's almost certainly a missing dependency
that `--no-deps` skipped (e.g. `tokenizers`, `huggingface_hub`, `safetensors`) — paste the error and
I'll pin the exact missing package rather than pulling the full dependency tree blind.


In [ ]:
%%bash
cd /kaggle/working
if [ ! -d "FastV" ]; then
  git clone --depth 1 https://github.com/pkunlp-icler/FastV.git
fi
cd FastV/src/FastV/llava-hf/transformers
pip install -e . --no-deps -q
pip install -q "tokenizers<0.19,>=0.14" pillow accelerate bitsandbytes sentencepiece
python -c "import transformers; print('transformers version:', transformers.__version__)"

## 3. Smoke test — baseline vs. FastV on one image

Loads `llava-hf/llava-1.5-7b-hf` in 8-bit (fits comfortably on one T4's 16GB with eager attention
headroom to spare), runs it once with FastV disabled and once with FastV enabled (K=2, R=50%,
matching the paper's default), and prints both outputs so you can eyeball that FastV's output is
still coherent (it doesn't need to match word-for-word, just be a sane image description, not
gibberish or an early cutoff).


In [3]:
import torch
# from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from transformers import (
    AutoTokenizer,             # <-- Added
    CLIPImageProcessor,        # <-- Added
    LlavaProcessor,            # <-- Added
    LlavaForConditionalGeneration,
    BitsAndBytesConfig
)
from PIL import Image
import requests

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
bnb_config = BitsAndBytesConfig(load_in_8bit=True)
# processor = AutoProcessor.from_pretrained(MODEL_ID)

# --- THE FIX ---
# Load the components individually to bypass the updated config file
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False)
image_processor = CLIPImageProcessor.from_pretrained(MODEL_ID)
processor = LlavaProcessor(image_processor=image_processor, tokenizer=tokenizer)
# ----

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [5]:
prompt = "USER: <image>\nWhat is shown in this image? Answer in one sentence.\nASSISTANT:"
image = Image.open(requests.get("http://images.cocodataset.org/val2017/000000039769.jpg", stream=True).raw)

def load_model(fastv_config=None):
    kwargs = dict(quantization_config=bnb_config, attn_implementation="eager", device_map="auto")
    if fastv_config is not None:
        kwargs["fastv_config"] = fastv_config
    return LlavaForConditionalGeneration.from_pretrained(MODEL_ID, **kwargs)

print("Loading baseline (no FastV)...")
model = load_model(fastv_config=None)
inputs = processor(prompt, image, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=64, do_sample=False, use_cache=False)
print("BASELINE:", processor.decode(out[0], skip_special_tokens=True))
del model
torch.cuda.empty_cache()

fastv_config = {
    "use_fastv": True,
    "fastv_k": 2,
    "fastv_r": 0.5,
    "image_token_start_index": 5,
    "image_token_length": 576,
}
print("\nLoading FastV (K=2, R=50%)...")
model = load_model(fastv_config=fastv_config)
inputs = processor(prompt, image, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=64, do_sample=False, use_cache=False)
print("FASTV:   ", processor.decode(out[0], skip_special_tokens=True))
del model
torch.cuda.empty_cache()

Loading baseline (no FastV)...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

BASELINE: USER:  <image> 
What is shown in this image? Answer in one sentence.
ASSISTANT: Two cats sleeping on a couch.

Loading FastV (K=2, R=50%)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

FASTV:    USER:  <image> 
What is shown in this image? Answer in one sentence.
ASSISTANT: Two cats sleeping on a couch.


**Checkpoint:** if both outputs above are coherent sentences describing the image, continue.
If you got a CUDA OOM, try `load_in_4bit=True` in `BitsAndBytesConfig` instead of 8-bit, or restart
the kernel to clear memory before re-running. If you got an import/attribute error, paste it back —
it likely means the patched transformers fork's `LlavaForConditionalGeneration.forward` doesn't
match this exact `llava-hf/llava-1.5-7b-hf` checkpoint's config in some field, which I'd need to fix
in the patch.

## 4. POPE — data prep (subset, no gated downloads)

In [6]:
%%bash
mkdir -p /kaggle/working/data
cd /kaggle/working/data
if [ ! -d "POPE" ]; then
  git clone --depth 1 https://github.com/AoiDragon/POPE.git
fi
ls POPE/output/coco


coco_pope_adversarial.json
coco_pope_popular.json
coco_pope_random.json


Cloning into 'POPE'...


In [7]:
import json, random, os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import urllib.request

random.seed(0)
N_PER_SPLIT = 150  # ~150 questions per POPE split (random/popular/adversarial) = 450 total. Raise once the pipeline is confirmed working.

pope_dir = Path("/kaggle/working/data/POPE/output/coco")
splits = ["coco_pope_random.json", "coco_pope_popular.json", "coco_pope_adversarial.json"]

subset = {}
for split in splits:
    lines = [json.loads(l) for l in open(pope_dir / split) if l.strip()]
    # sample by unique image to keep the image-download set small, then take all questions for those images
    images = sorted(set(l["image"] for l in lines))
    random.shuffle(images)
    keep_images = set(images[: max(1, N_PER_SPLIT // 6)])  # ~6 questions/image on average in POPE
    sampled = [l for l in lines if l["image"] in keep_images][:N_PER_SPLIT]
    subset[split] = sampled
    print(split, "->", len(sampled), "questions over", len(keep_images), "images")

img_dir = Path("/kaggle/working/data/coco_val2014_subset")
img_dir.mkdir(exist_ok=True)
all_images = sorted(set(l["image"] for s in subset.values() for l in s))
print(f"\nDownloading {len(all_images)} unique COCO val2014 images...")

def fetch(fname):
    dest = img_dir / fname
    if dest.exists():
        return
    url = f"http://images.cocodataset.org/val2014/{fname}"
    urllib.request.urlretrieve(url, dest)

with ThreadPoolExecutor(max_workers=16) as ex:
    list(ex.map(fetch, all_images))

print("Done. Downloaded:", len(list(img_dir.glob('*.jpg'))))


coco_pope_random.json -> 150 questions over 25 images
coco_pope_popular.json -> 150 questions over 25 images
coco_pope_adversarial.json -> 150 questions over 25 images

Done. Downloaded: 73


## 5. POPE — run baseline + FastV inference

Runs the sampled POPE questions through baseline and a couple of FastV (K, R) settings. This
reimplements POPE's own metric (accuracy / precision / recall / F1 / yes-ratio) directly rather than
depending on LLaVA's `eval.zip` (which sits behind a Google Drive link) — POPE's ground-truth format
is simple enough (`yes`/`no` labels) that this is a faithful, self-contained reproduction.


In [11]:
import re, gc, torch
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from PIL import Image

# processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False)
image_processor = CLIPImageProcessor.from_pretrained(MODEL_ID)
processor = LlavaProcessor(image_processor=image_processor, tokenizer=tokenizer)

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

all_questions = [l for s in subset.values() for l in s]
print("Total POPE questions in this run:", len(all_questions))

def parse_yes_no(text):
    text = text.split("ASSISTANT:")[-1].strip().lower()
    if text.startswith("yes"):
        return "yes"
    if text.startswith("no"):
        return "no"
    # fallback: search for first yes/no token
    m = re.search(r"\b(yes|no)\b", text)
    return m.group(1) if m else "no"

def run_eval(fastv_config, tag, max_examples=None):
    kwargs = dict(quantization_config=bnb_config, attn_implementation="eager", device_map="auto")
    if fastv_config is not None:
        kwargs["fastv_config"] = fastv_config
    model = LlavaForConditionalGeneration.from_pretrained(MODEL_ID, **kwargs)
    model.eval()

    preds, labels = [], []
    qs = all_questions[:max_examples] if max_examples else all_questions
    for i, q in enumerate(qs):
        img = Image.open(img_dir / q["image"]).convert("RGB")
        prompt = f"USER: <image>\n{q['text']}\nASSISTANT:"
        inputs = processor(prompt, img, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=10, do_sample=False, use_cache=False)
        pred = parse_yes_no(processor.decode(out[0], skip_special_tokens=True))
        preds.append(pred)
        labels.append(q["label"])
        if (i + 1) % 50 == 0:
            print(f"  [{tag}] {i+1}/{len(qs)}")

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return preds, labels

def pope_metrics(preds, labels):
    tp = sum(p == "yes" and l == "yes" for p, l in zip(preds, labels))
    fp = sum(p == "yes" and l == "no" for p, l in zip(preds, labels))
    tn = sum(p == "no" and l == "no" for p, l in zip(preds, labels))
    fn = sum(p == "no" and l == "yes" for p, l in zip(preds, labels))
    acc = (tp + tn) / len(labels)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    yes_ratio = preds.count("yes") / len(preds)
    return dict(accuracy=acc, precision=prec, recall=rec, f1=f1, yes_ratio=yes_ratio)

results = {}

print("=== Baseline (no pruning) ===")
preds, labels = run_eval(None, "baseline")
results["baseline"] = pope_metrics(preds, labels)

for k, r in [(2, 0.5), (2, 0.75), (3, 0.5)]:
    tag = f"FastV K={k} R={r}"
    print(f"=== {tag} ===")
    cfg = {"use_fastv": True, "fastv_k": k, "fastv_r": r,
           "image_token_start_index": 5, "image_token_length": 576}
    preds, labels = run_eval(cfg, tag)
    results[tag] = pope_metrics(preds, labels)

print("\n\n==== POPE RESULTS SUMMARY ====")
print(f"{'setting':<20}{'acc':>8}{'f1':>8}{'yes%':>8}")
for tag, m in results.items():
    print(f"{tag:<20}{m['accuracy']*100:>7.1f}%{m['f1']*100:>7.1f}%{m['yes_ratio']*100:>7.1f}%")


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Total POPE questions in this run: 450
=== Baseline (no pruning) ===


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  [baseline] 50/450
  [baseline] 100/450
  [baseline] 150/450
  [baseline] 200/450
  [baseline] 250/450
  [baseline] 300/450
  [baseline] 350/450
  [baseline] 400/450
  [baseline] 450/450
=== FastV K=2 R=0.5 ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  [FastV K=2 R=0.75] 50/450
  [FastV K=2 R=0.75] 100/450
  [FastV K=2 R=0.75] 150/450
  [FastV K=2 R=0.75] 200/450
  [FastV K=2 R=0.75] 250/450
  [FastV K=2 R=0.75] 300/450
  [FastV K=2 R=0.75] 350/450
  [FastV K=2 R=0.75] 400/450
  [FastV K=2 R=0.75] 450/450
=== FastV K=3 R=0.5 ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  [FastV K=3 R=0.5] 50/450
  [FastV K=3 R=0.5] 100/450
  [FastV K=3 R=0.5] 150/450
  [FastV K=3 R=0.5] 200/450
  [FastV K=3 R=0.5] 250/450
  [FastV K=3 R=0.5] 300/450
  [FastV K=3 R=0.5] 350/450
  [FastV K=3 R=0.5] 400/450
  [FastV K=3 R=0.5] 450/450


==== POPE RESULTS SUMMARY ====
setting                  acc      f1    yes%
baseline               88.7%   88.7%   50.2%
FastV K=2 R=0.5        87.6%   87.0%   46.0%
FastV K=2 R=0.75       83.6%   81.2%   37.6%
FastV K=3 R=0.5        88.0%   87.6%   46.4%


**What to check:** baseline accuracy on POPE (LLaVA-1.5-7B) should land roughly in the mid-to-high
80s% F1 on the `random` split (higher) down to mid-80s on `adversarial` (harder) in the published
literature — since we're mixing all three splits into one pooled subset here for a quick first pass,
just check the numbers are in a sane ballpark (not near 50%, which would mean something's broken,
e.g. all-yes or all-no predictions) rather than expecting an exact published-number match. If FastV's
F1 stays close to baseline while `yes_ratio`/prediction pattern looks reasonable at R=50-75%, that
matches the paper's claim. **Send me this summary table and I'll help interpret it / debug if
anything looks off.**

Once this subset (~450 questions) runs cleanly, bump `N_PER_SPLIT` up (e.g. to 500-1000) for a more
statistically solid number, budgeting more Kaggle GPU time accordingly.

## 6. GQA (optional — larger download, run after POPE works)

GQA doesn't offer per-image download URLs the way COCO does, so this needs the full `images.zip`
(~20GB) from the official GQA site — expect this cell to take a while depending on Kaggle's network
speed. Kaggle's `/kaggle/working` scratch space allows this temporarily even though you can't persist
20GB to output. Only run this once POPE above is working.

In [ ]:
%%bash
mkdir -p /kaggle/working/data/gqa
cd /kaggle/working/data/gqa
# Questions (small)
[ -f questions1.2.zip ] || wget -q https://downloads.cs.stanford.edu/nlp/data/gqa/questions1.2.zip
unzip -oq questions1.2.zip -d questions
ls questions | head
# Images (large, ~20GB) -- only uncomment when ready
# [ -f images.zip ] || wget -q https://downloads.cs.stanford.edu/nlp/data/gqa/images.zip
# unzip -oq images.zip -d images


In [ ]:
import json, random
random.seed(0)
with open("/kaggle/working/data/gqa/questions/testdev_balanced_questions.json") as f:
    gqa_q = json.load(f)

qids = list(gqa_q.keys())
random.shuffle(qids)
gqa_subset = {qid: gqa_q[qid] for qid in qids[:300]}
print("Sampled", len(gqa_subset), "GQA testdev questions.")
print("Unique images referenced:", len(set(v['imageId'] for v in gqa_subset.values())))
print("\nUncomment the images.zip download above once you're ready to run this section,")
print("then adapt the inference loop from Section 5 (same pattern: build prompt, generate, parse answer).")


## 7. MME — blocked pending manual access

MME's benchmark images are gated behind a request process via the official repo
([BradyFU/Awesome-Multimodal-Large-Language-Models](https://github.com/BradyFU/Awesome-Multimodal-Large-Language-Models/tree/Evaluation))
rather than a direct download link — historically this has meant filling out a form / emailing the
maintainers for the image archive. **I couldn't verify the exact current process from here (my
sandbox can't fill out forms), so please check that repo's README directly.** Once you have the
images, the inference/scoring pattern is the same as POPE above (MME uses yes/no questions across 14
subtasks) — come back and I'll adapt the eval loop once you have the data. Starting with GQA + POPE
first is not a problem; POPE already gives you the hallucination-relevant metric you care about most.


In [12]:
import torch
import re
import gc
from PIL import Image
from datasets import load_dataset
from collections import defaultdict

# 1. Load MME Dataset
print("Loading MME dataset from Hugging Face...")
mme_dataset = load_dataset("lmms-lab-encoder/MME", split="test")

# 2. Answer Parsing Helper
def parse_mme_answer(text):
    text = text.split("ASSISTANT:")[-1].strip().lower()
    if text.startswith("yes"):
        return "yes"
    if text.startswith("no"):
        return "no"
    m = re.search(r"\b(yes|no)\b", text)
    return m.group(1) if m else "no"

# 3. Model Loading Helper (includes the device_map fix for 8-bit)
def load_model(fastv_config=None):
    kwargs = dict(
        quantization_config=bnb_config,
        attn_implementation="eager",
        device_map="auto" 
    )
    if fastv_config is not None:
        kwargs["fastv_config"] = fastv_config
    return LlavaForConditionalGeneration.from_pretrained(MODEL_ID, **kwargs)

# 4. Evaluation Loop
def eval_mme(fastv_config, tag, max_examples=None):
    print(f"\n--- Starting Evaluation: {tag} ---")
    model = load_model(fastv_config)
    model.eval()
    
    results_by_category = defaultdict(lambda: {"correct": 0, "total": 0})
    samples = mme_dataset if max_examples is None else mme_dataset.shuffle(seed=0).select(range(max_examples))
    
    for i, item in enumerate(samples):
        image = item["image"].convert("RGB")
        question_text = item["question"]
        ground_truth = item["answer"].strip().lower()
        category = item["category"]
        
        prompt = f"USER: <image>\n{question_text}\nASSISTANT:"
        
        # Send inputs to cuda directly to avoid .to() errors on the 8-bit model
        inputs = processor(prompt, image, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=10, do_sample=False, use_cache=False)
            
        pred = parse_mme_answer(processor.decode(out[0], skip_special_tokens=True))
        
        if pred == ground_truth:
            results_by_category[category]["correct"] += 1
        results_by_category[category]["total"] += 1
        
        if (i + 1) % 100 == 0:
            print(f"  Processed {i + 1}/{len(samples)} questions...")
            
    # Cleanup to prevent CUDA Out Of Memory between runs
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    # Calculate and print metrics
    print(f"\n==== MME RESULTS SUMMARY: {tag} ====")
    metrics = {}
    total_correct = 0
    total_questions = 0
    for cat, data in results_by_category.items():
        acc = data["correct"] / data["total"] if data["total"] > 0 else 0.0
        metrics[cat] = acc
        total_correct += data["correct"]
        total_questions += data["total"]
        print(f"{cat.ljust(20)}: {acc*100:.2f}% ({data['correct']}/{data['total']})")
        
    overall_acc = total_correct / total_questions if total_questions > 0 else 0.0
    print(f"{'OVERALL'.ljust(20)}: {overall_acc*100:.2f}% ({total_correct}/{total_questions})")
    return metrics

# 5. Execute Runs 
# Note: MME has ~2.37k rows. Set MAX_EX to None to run the full dataset, 
# or keep it small (e.g., 200) to verify the loop works first.
MAX_EX = 200 

baseline_metrics = eval_mme(
    fastv_config=None, 
    tag="Baseline", 
    max_examples=MAX_EX
)

fastv_config_50 = {
    "use_fastv": True,
    "fastv_k": 2,
    "fastv_r": 0.5,
    "image_token_start_index": 5,
    "image_token_length": 576,
}

fastv_metrics = eval_mme(
    fastv_config=fastv_config_50, 
    tag="FastV (R=0.5)", 
    max_examples=MAX_EX
)

Loading MME dataset from Hugging Face...


README.md:   0%|          | 0.00/493 [00:00<?, ?B/s]

data/test-00000-of-00004-a25dbe3b44c4fda(…):   0%|          | 0.00/34.9M [00:00<?, ?B/s]

data/test-00001-of-00004-7d22c7f1aba6fca(…):   0%|          | 0.00/110M [00:00<?, ?B/s]

data/test-00002-of-00004-594798fd3f5b029(…):   0%|          | 0.00/695M [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  Processed 100/200 questions...
  Processed 200/200 questions...

==== MME RESULTS SUMMARY: Baseline ====
code_reasoning      : 57.50% (23/40)
artwork             : 75.62% (121/160)
OVERALL             : 72.00% (144/200)

--- Starting Evaluation: FastV (R=0.5) ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  Processed 100/200 questions...
  Processed 200/200 questions...

==== MME RESULTS SUMMARY: FastV (R=0.5) ====
code_reasoning      : 55.00% (22/40)
artwork             : 75.62% (121/160)
OVERALL             : 71.50% (143/200)


## Summary — what to send back

1. Section 3 smoke-test outputs (baseline + FastV sentences)
2. Section 5's `==== POPE RESULTS SUMMARY ====` table
3. Any traceback, if something errored, pasted in full

With that I can tell you whether the numbers look right, help debug anything that broke, and then
we move to porting the InfoScore layer-selection logic on top of this same setup (Phase 1).